# 01 — Ingestão e Exploração dos Dados

**Fonte**: healthbr-data (SI-PNI em Parquet via S3 público)

**Período**: maio/2025 – maio/2026

**Foco**: Vacinas contra Influenza

In [ ]:
import sys
sys.path.append('..')

import pyarrow.dataset as pds
import pyarrow.fs as fs
import pyarrow.compute as pc
import pandas as pd
import polars as pl

from src.config import S3_ENDPOINT, S3_ACCESS_KEY, S3_SECRET_KEY, S3_REGION, S3_BUCKET, ANOS, COLUNAS_PNI

In [ ]:
# Conectar ao S3
s3 = fs.S3FileSystem(
    endpoint_override=S3_ENDPOINT,
    access_key=S3_ACCESS_KEY,
    secret_key=S3_SECRET_KEY,
    region=S3_REGION,
)

dataset = pds.dataset(
    S3_BUCKET,
    filesystem=s3,
    format="parquet",
    partitioning="hive",
)

print(f"Partições disponíveis: {len(list(dataset.get_fragments()))}")

In [ ]:
# Carregar dados de 2025 e 2026
table = dataset.to_table(
    filter=pds.field("ano").isin(ANOS),
    columns=COLUNAS_PNI,
)

print(f"Registros carregados: {table.num_rows:,}")
print(f"Colunas: {table.column_names}")

In [ ]:
# Filtrar apenas Influenza
mask = pc.match_substring(pc.upper(table.column("ds_vacina")), "INFLUENZA")
mask = pc.or_(mask, pc.match_substring(pc.upper(table.column("ds_vacina")), "GRIPE"))
table_influenza = table.filter(mask)

print(f"Registros de Influenza: {table_influenza.num_rows:,}")

In [ ]:
# Amostra dos dados
df_sample = table_influenza.to_pandas().head(100)
df_sample.head(10)

In [ ]:
# Verificar vacinas encontradas
print("ds_vacina únicos:")
for v in sorted(table_influenza.column("ds_vacina").unique().to_pylist()):
    print(f"  - {v}")

In [ ]:
# Distribuição por UF
uf_counts = (
    table_influenza.column("sg_uf")
    .value_counts()
    .to_pandas()
)
uf_counts.columns = ["sg_uf", "total_doses"]
uf_counts

In [ ]:
# Download SIH (exemplo: uma UF)
from pysus.online_data.SIH import download

df_sih_sp = download("SP", 2025)
print(f"SIH SP/2025: {len(df_sih_sp)} registros")
df_sih_sp.head(2)

In [ ]:
# Download SIM (exemplo: uma UF)
from pysus.online_data.SIM import download

df_sim_sp = download("SP", 2025)
print(f"SIM SP/2025: {len(df_sim_sp)} registros")
df_sim_sp.head(2)

---
**Próximo notebook**: `02_transform.ipynb` — limpeza e agregação